In [1]:
import stackstac
import pystac_client
from jupytergis.tiler import GISDocument

doc = GISDocument()

doc.add_raster_layer(
    url="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    name="Google Satellite",
    attribution="Google",
    opacity=0.6,
)


'22eb8834-e72d-4d4d-aa67-a036fc656065'

In [2]:
URL = "https://earth-search.aws.element84.com/v1"
catalog = pystac_client.Client.open(URL)
lon, lat = -106.6142, 35.1789

In [3]:
%%time
items = catalog.search(
    intersects=dict(type="Point", coordinates=[lon, lat]),
    collections=["sentinel-2-l2a"],
    datetime="2020-03-01/2020-06-01"
).item_collection()
len(items)

CPU times: user 105 ms, sys: 5.95 ms, total: 111 ms
Wall time: 7.82 s


147

In [4]:
%time stack = stackstac.stack(items, epsg=4326, bounds_latlon=[-106.763, 35.014, -106.492, 35.246])

CPU times: user 95.1 ms, sys: 8.76 ms, total: 104 ms
Wall time: 104 ms


In [5]:
lowcloud = stack[stack["eo:cloud_cover"] < 20]
rgb = lowcloud.sel(band=["red", "green", "blue"])
monthly = rgb.resample(time="MS").median("time", keep_attrs=True)

In [13]:
import dask.diagnostics
with dask.diagnostics.ProgressBar():
    data = monthly.compute()

[########################################] | 100% Completed | 216.53 s


In [14]:
vmin_acc, vmax_acc = int(data.min().compute()), int(data.max().compute())
vmin_acc, vmax_acc

(0, 1)

In [15]:
out = data.isel(time=0, band=0)


In [17]:
await doc.add_tiler_layer(
    name="red",
    data_array=out,
    colormap_name="plasma",
    rescale=(vmin_acc, vmax_acc),
    reproject="max",
)

'4ec8529e-0005-4fed-b970-4d739769f55f'

In [8]:
doc

In [ ]:
data.plot.imshow(row="time", rgb="band", robust=True, size=6);


In [109]:
aoi

<xarray.DataArray 'stackstac-ca5972694268cb8722a54da6ee9587b7' (time: 3,
                                                                band: 3,
                                                                y: 2556, x: 4587)> Size: 844MB
dask.array<stack, shape=(3, 3, 2556, 4587), dtype=float64, chunksize=(1, 1, 639, 639), chunktype=numpy.ndarray>
Coordinates: (12/19)
  * time                                     (time) datetime64[ns] 24B 2020-0...
  * band                                     (band) <U12 144B 'red' ... 'blue'
    raster:bands                             (band) object 24B None None None
    title                                    (band) <U31 372B 'Red (band 4) -...
    gsd                                      (band) object 24B 10 10 10
    common_name                              (band) object 24B 'red' ... 'blue'
    ...                                       ...
    s2:datatake_type                         <U8 32B 'INS-NOBS'
    constellation                            <U10 40B 'sentinel-2'
    mgrs:utm_zone                            int64 8B 13
    s2:product_type                          <U7 28B 'S2MSI2A'
    proj:code                                <U10 40B 'EPSG:32613'
    epsg                                     int64 8B 4326
Attributes:
    spec:           RasterSpec(epsg=4326, bounds=(-106.76302354351425, 35.013...
    crs:            epsg:4326
    transform:      | 0.00, 0.00,-106.76|\n| 0.00,-0.00, 35.25|\n| 0.00, 0.00...
    resolution_xy:  (5.9090512048020614e-05, 9.07906787896377e-05)